In [22]:
import os
import string
import annoy
import codecs

from pymorphy3 import MorphAnalyzer
from stop_words import get_stop_words
from gensim.models import Word2Vec

import numpy as np
from tqdm.std import tqdm
import pandas as pd

Предобработаем ответы mail.ru из файла: к каждому вопросу присоединим 1 ответ и запишем в файл на будущее. Это позволит нам сэкономить время и ресурсы при дальнейшем препроцессинге текста

In [54]:
question = None
written = False

#Мы идем по всем записям, берем первую строку как вопрос
# и после знака --- находим ответ
with codecs.open("data/prepared_answers.txt","w", "utf-8") as fout:
    with codecs.open("data/Otvety.txt", "r", "utf-8") as fin:
        for line in tqdm(fin):
            if line.startswith("---"):
                written = False
                continue
            if not written and question is not None:
                fout.write(question.replace("\t", " ").strip() + "\t" + line.replace("\t", " "))
                written = True
                question = None
                continue
            if not written:
                question = line.strip()
                continue

7551302it [00:25, 294412.47it/s]


Теперь нам нужно предобработать текст, чтобы обучить word2vec и получить эмбеддинги. Удаляем знаки препинания и делаем лемматизацию

In [24]:
def preprocess_txt(line):
    spls = "".join(i for i in line.strip() if i not in exclude).split()
    spls = [morpher.parse(i.lower())[0].normal_form for i in spls]
    spls = [i for i in spls if i not in sw and i != ""]
    return spls

In [25]:
sentences = []

morpher = MorphAnalyzer()
sw = set(get_stop_words("ru"))
exclude = set(string.punctuation)
c = 0

with codecs.open("data/Otvety.txt", "r", "utf-8") as fin:
    for line in tqdm(fin):
        spls = preprocess_txt(line)
        sentences.append(spls)
        c += 1
        if c > 500000:
            break

500000it [11:26, 727.84it/s] 


In [ ]:
# Обучим модель word2vec на наших вопросах
sentences = [i for i in sentences if len(i) > 2]
model = Word2Vec(sentences=sentences, vector_size=100, min_count=1, window=5)
model.save("w2v_model")

Теперь нам нужно сложить в индекс все вопросы. Используем библиотеку annoy. Проходимся по всем ответам, считаем, что вектор предложения - сумма word2vecов слов, которые входят в него (конечно же усредненная)

In [55]:
index = annoy.AnnoyIndex(100 ,'angular')

index_map = {}
counter = 0

with codecs.open("data/prepared_answers.txt", "r", "utf-8") as f:
    for line in tqdm(f):
        n_w2v = 0
        spls = line.split("\t")
        if len(spls) < 2:  # Пропускаем некорректные строки
            continue
        question = preprocess_txt(spls[0])
        vector = np.zeros(100)
        for word in question:
            if word in model.wv:
                vector += model.wv[word]
                n_w2v += 1
        if n_w2v > 0:  # Добавляем только если нашли хотя бы одно слово
            vector = vector / n_w2v
            index_map[counter] = spls[1]
            index.add_item(counter, vector)
            counter += 1

index.build(10)
index.save('speaker.ann')

=== ПЕРЕСОЗДАНИЕ ИНДЕКСА ===
Добавляем вопросы в индекс...


10124it [00:14, 708.48it/s]


Добавлено 10000 вопросов
Строим индекс...
Индекс сохранен
Тестируем новый индекс...
Тест 0: [3]
Тест 1: [3]
Тест 2: [3]
Новый индекс готов: 10000 элементов


Теперь остается реализовать метод, который получит на вход вопрос и найдет ответ к нему!
Мы препроцессим вопрос, находим ближайший вопрос и выбираем ответ на ближайший вопрос.

In [40]:
def find_answer(question):
    preprocessed_question = preprocess_txt(question)
    print(f"Предобработанный вопрос: {preprocessed_question}")

    n_w2v = 0
    vector = np.zeros(100)
    for word in preprocessed_question:
        if word in model.wv:
            vector += model.wv[word]
            n_w2v += 1

    print(f"Найдено слов в модели: {n_w2v}")
    if n_w2v > 0:
        vector = vector / n_w2v

    answer_index = index.get_nns_by_vector(vector, 1)
    print(f"Найден ID: {answer_index[0]}")
    return index_map[answer_index[0]]

In [51]:
find_answer("Как парни относятся к цветным линзам?")

Предобработанный вопрос: ['парень', 'относиться', 'цветной', 'линза']
Найдено слов в модели: 4
Найден ID: 3


'4. Охотники и Жертвы, Ледяной укус, Поцелуй тьмы, Кровная клятва. \n'